# PaddleOCR UI — Jupyter 一键调用
填写文件路径后，按 `Shift+Enter` 逐格运行即可。
输出 ZIP 文件生成在源文件同目录下。

In [ ]:
# ═══════════════════════════════════════
# 第 1 步：初始化环境
# ═══════════════════════════════════════

import os, sys, logging
from datetime import datetime

# 加载环境变量
env_file = '/mnt/workspace/env/paddleocr-ui.sh'
if os.path.exists(env_file):
    with open(env_file) as f:
        for line in f:
            if line.startswith('export '):
                kv = line.strip()[7:].split('=', 1)
                if len(kv) == 2:
                    os.environ[kv[0]] = kv[1].strip('"').strip("'")

# 将项目路径加入 sys.path
PROJ = '/mnt/workspace/project/paddleocr-ui'
if PROJ not in sys.path:
    sys.path.insert(0, PROJ)

os.environ.setdefault('PADDLE_PDX_MODEL_SOURCE', 'modelscope')
os.environ.setdefault('PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK', 'True')

logging.basicConfig(level=logging.WARN)

print('✅ 环境就绪')
print(f'   PYTHONPATH: {os.environ.get("PYTHONPATH", "未设置")}')

In [ ]:
# ═══════════════════════════════════════
# 第 2 步：填写要 OCR 的文件路径（支持 PDF / JPG / PNG）
# ═══════════════════════════════════════

# 👇 在这里填写你的文件路径
FILE_PATH = ""  # ← 改成你的文件路径，如 "/home/user/contract.pdf"

# 检查文件
if not FILE_PATH:
    print("⚠ 请先填写 FILE_PATH 再运行此格")
elif not os.path.exists(FILE_PATH):
    print(f"❌ 文件不存在: {FILE_PATH}")
else:
    print(f"📄 {FILE_PATH}")
    print(f"   大小: {os.path.getsize(FILE_PATH)/1024/1024:.1f} MB")
    print(f"   类型: {os.path.splitext(FILE_PATH)[1]}")

In [ ]:
# ═══════════════════════════════════════
# 第 3 步：配置 OCR 参数（可跳过，使用默认值）
# ═══════════════════════════════════════

SETTINGS = {
    "dpi": 250,               # 扫描件渲染精度 (150/250/400)
    "lang": "ch",            # 语言 (ch=中文 / en=英文)
    "table": 1,              # 表格检测 (1=开启 / 0=关闭)
    "table_strategy": "lines", # 表格策略 (lines/text/auto)
    "table_merge": 0,        # 跨页合并 (1=开启 / 0=关闭)
    "stamp": 1,             # 印章检测 (1=开启 / 0=关闭)
    "image": 0,             # 图片识别强度 (0/1/2)
    "extract_images": 0,    # 单独导出图片 (1=开启 / 0=关闭)
}

print("当前参数:")
for k, v in SETTINGS.items():
    print(f"  {k}: {v}")

In [ ]:
# ═══════════════════════════════════════
# 第 4 步：执行 OCR（较慢，请耐心等待）
# ═══════════════════════════════════════

import shutil, tempfile, zipfile
from pathlib import Path
from app.pipeline.core.pipeline import process_pdf
from app.pipeline.export.word_writer import write_word
from app.pipeline.export.excel_writer import write_excel
from app.pipeline.export.text_writer import write_text
from app.pipeline.export.archiver import create_archive

if not FILE_PATH or not os.path.exists(FILE_PATH):
    print("❌ 请先在第 2 步填写有效的 FILE_PATH")
else:
    src_dir = os.path.dirname(os.path.abspath(FILE_PATH))
    src_name = os.path.basename(FILE_PATH)
    
    # 临时工作目录
    work_dir = tempfile.mkdtemp(prefix='ocr_')
    print(f"工作目录: {work_dir}")
    
    # 运行管线
    print("正在处理...")
    doc_layout = process_pdf(FILE_PATH, work_dir, **SETTINGS)
    print(f"✅ 完成: {len(doc_layout.pages)} 页, {len(doc_layout.tables)} 张表")
    
    # 生成中间文件
    write_word(doc_layout, os.path.join(work_dir, "output.docx"))
    write_excel(doc_layout.tables, os.path.join(work_dir, "tables.xlsx"))
    write_text(doc_layout, os.path.join(work_dir, "output.txt"))
    
    # 打包 ZIP
    zip_path = create_archive(doc_layout, work_dir, "jupyter", src_name)
    
    # 拷贝 ZIP 到源文件目录
    final_zip = os.path.join(src_dir, os.path.basename(zip_path))
    shutil.copy2(zip_path, final_zip)
    
    # 清理临时目录
    shutil.rmtree(work_dir, ignore_errors=True)
    
    print(f"\n📦 ZIP 已生成: {final_zip}")
    print(f"   大小: {os.path.getsize(final_zip)/1024/1024:.1f} MB")
    print(f"   内容: .docx + .xlsx + .txt")